In [7]:
import pandas as pd
import geopandas as gpd
import requests
from urllib.parse import urlencode

In [42]:
# 결과 저장용 리스트
rows = []

In [43]:
region = [11, 12, 13, 14, 15, 16, 17, 18, 21, 22, 23, 24, 25, 26, 27, 28, 29]

In [44]:
for r in region:
    request_url = f'https://mediboard.nemc.or.kr/api/v1/search/handy?searchCondition=regional&emogloca={r}'
    response = requests.get(request_url)

    if response.status_code == 200:
        data = response.json()
        for item in data['result']['data']:
            rows.append({
                'emogCode': item['emogCode'],
                'emergencyRoomNickname': item['emergencyRoomNickname'],
                'emergencyRoomName': item['emergencyRoomName'],
                'latitude': item['latitude'],
                'longitude': item['longitude'],
                'address': item['address'],
                'childEmergencyAvailable': item['childEmergencyAvailable'],
                'childEmergencyTotal': item['childEmergencyTotal'],
                'availabilityStatus': 2  # 초기값 2
            })
    else:
        print(f"지역 코드 {r} 요청 실패: {response.status_code}")

In [45]:
df = pd.DataFrame(rows)

In [49]:
df.columns

Index(['emogCode', 'emergencyRoomNickname', 'emergencyRoomName', 'latitude',
       'longitude', 'address', 'childEmergencyAvailable',
       'childEmergencyTotal', 'availabilityStatus'],
      dtype='object')

In [50]:
# It is possible to create a geometry column with the given coordinates
# gpd.points_from_xy is a function to create points based on given coordinates

emergency_gdf = gpd.GeoDataFrame(df, 
                                # a function to create points based on given coordinates
                                geometry=gpd.points_from_xy(df.longitude, df.latitude) 
                               )
emergency_gdf

,emogCode,emergencyRoomNickname,emergencyRoomName,latitude,longitude,address,childEmergencyAvailable,childEmergencyTotal,availabilityStatus,geometry
0,A1100001,경희대,경희대학교병원,37.593877,127.051832,서울특별시 동대문구 경희대로 23 (회기동),3,3,2,POINT (127.05183 37.59388)
1,A1100002,건국대,건국대학교병원,37.540845,127.072123,서울특별시 광진구 능동로 120-1 (화양동),0,0,2,POINT (127.07212 37.54084)
2,A1100003,중앙대,중앙대학교병원,37.507074,126.960794,서울특별시 동작구 흑석로 102 (흑석동),4,4,2,POINT (126.96079 37.50707)
3,A1100004,순천향서울,순천향대학교 부속 서울병원,37.533842,127.004418,서울특별시 용산구 대사관로 59 (한남동),4,4,2,POINT (127.00442 37.53384)
4,A1100005,이대목동,이화여자대학교의과대학부속목동병원,37.536543,126.886216,서울특별시 양천구 안양천로 1071 (목동),6,8,2,POINT (126.88622 37.53654)
...,...,...,...,...,...,...,...,...,...,...
411,A2900002,제주특별자치도서귀포의료원,제주특별자치도서귀포의료원,33.255355,126.564201,제주특별자치도 서귀포시 장수로 47 (동홍동),4,4,2,POINT (126.5642 33.25535)
412,A2900003,제주한국,의료법인 혜인의료재단 한국병원,33.500186,126.516989,제주특별자치도 제주시 서광로 193 (삼도일동),None,None,2,POINT (126.51699 33.50019)
413,A2900005,한마음병원,한마음병원,33.496261,126.546225,제주특별자치도 제주시 연신로 52 (이도이동),None,None,2,POINT (126.54622 33.49626)
414,A2900006,제주중앙,의료법인중앙의료재단중앙병원,33.492346,126.470587,"제주특별자치도 제주시 월랑로 91-0 (이호이동,중앙병원)",1,1,2,POINT (126.47059 33.49235)


In [51]:
print(emergency_gdf.crs)

None


In [52]:
# .set_crs() method is used to set the CRS of the GeoDataFrame
emergency_gdf = emergency_gdf.set_crs(epsg=4326)


In [53]:
emergency_gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [54]:
# With the correct crs, the .explore() method works
emergency_gdf.explore()

In [55]:
emergency_gdf.columns

Index(['emogCode', 'emergencyRoomNickname', 'emergencyRoomName', 'latitude',
       'longitude', 'address', 'childEmergencyAvailable',
       'childEmergencyTotal', 'availabilityStatus', 'geometry'],
      dtype='object')

In [56]:
emergency_gdf.to_file("emergency.geojson", driver="GeoJSON", encoding="utf-8")
